# Honegumi for Materials Bayesian Optimization

This notebook showcases two related tools:

1. **Honegumi** for deterministic template generation of Ax-based Bayesian optimization code.
2. **Honegumi RAG Assistant** for turning a natural-language problem description into Honegumi parameters and then into a fuller Bayesian optimization script.

The goal is to make it easy to go from a materials problem statement to a runnable starting point.


In [1]:
from pathlib import Path
from pprint import pprint

import ipywidgets as widgets
import pandas as pd
from IPython.display import Code, Markdown, display

import honegumi
from honegumi.ax._ax import option_rows
from honegumi.ax.utils import constants as cst
from honegumi.core._honegumi import Honegumi
from honegumi_rag_assistant.app_config import settings
from honegumi_rag_assistant.extractors import ParameterExtractor
from honegumi_rag_assistant.orchestrator import run_from_text

script_template_dir = honegumi.ax.__path__[0]
core_template_dir = honegumi.core.__path__[0]

hg = Honegumi(
    cst,
    option_rows,
    script_template_dir=script_template_dir,
    core_template_dir=core_template_dir,
    script_template_name="main.py.jinja",
    core_template_name="honegumi.html.jinja",
)

generated_dir = Path("generated")
generated_dir.mkdir(exist_ok=True)
settings.reload_from_env()


## 1. The Honegumi Option Space

Honegumi exposes a finite set of high-level switches that describe the shape of the Bayesian optimization problem.
That makes it especially useful for teaching because students can think in terms of:

- single vs. multi-objective,
- existing data vs. greenfield optimization,
- batch vs. sequential evaluation,
- categorical variables and constraints.


In [2]:
options_df = pd.DataFrame(
    [
        {
            "name": row["name"],
            "display_name": row["display_name"],
            "options": row["options"],
            "hidden": row["hidden"],
            "disabled": row["disable"],
        }
        for row in option_rows
    ]
)
display(options_df)


,name,display_name,options,hidden,disabled
0,objective,Objective,"[Single, Multi]",False,False
1,model,Model Type,"[Default, Custom, Fully Bayesian]",False,False
2,task,Task,"[Single, Multi]",False,False
3,categorical,Categorical Parameter,"[False, True]",False,False
4,custom_gen,Custom Generator,"[False, True]",True,False
5,sum_constraint,Sum Constraint,"[False, True]",False,False
6,order_constraint,Order Constraint,"[False, True]",False,False
7,linear_constraint,Linear Constraint,"[False, True]",False,False
8,composition_constraint,Composition Constraint,"[False, True]",False,False
9,custom_threshold,Custom Threshold,"[False, True]",False,False


### Quick Interpretation Guide

Honegumi is helpful in class because each switch corresponds to a design decision that students can reason about before writing code.


In [3]:
decision_guide = pd.DataFrame(
    [
        {"Decision": "objective", "Teaching question": "Are we optimizing one materials property or balancing several at once?"},
        {"Decision": "existing_data", "Teaching question": "Do we already have historical experiments to warm-start the model?"},
        {"Decision": "categorical", "Teaching question": "Is one of the variables a discrete choice like solvent or substrate?"},
        {"Decision": "composition_constraint", "Teaching question": "Do the fractions need to add up to a total composition?"},
        {"Decision": "synchrony", "Teaching question": "Can the lab evaluate one candidate at a time or a batch in parallel?"},
    ]
)
display(decision_guide)


,Decision,Teaching question
0,objective,Are we optimizing one materials property or ba...
1,existing_data,Do we already have historical experiments to w...
2,categorical,Is one of the variables a discrete choice like...
3,composition_constraint,Do the fractions need to add up to a total com...
4,synchrony,Can the lab evaluate one candidate at a time o...


## 2. Deterministic Template Generation with Honegumi

We will start with a materials-flavored use case:

- optimize a cathode formulation,
- use existing screening data,
- enforce composition-aware reasoning,
- run experiments in small parallel batches.


In [4]:
manual_problem = {
    "objective": "Single",
    "model": "Default",
    "task": "Single",
    "categorical": False,
    "sum_constraint": False,
    "order_constraint": False,
    "linear_constraint": False,
    "composition_constraint": True,
    "custom_threshold": False,
    "existing_data": True,
    "synchrony": "Batch",
    "visualize": True,
}

options_model = hg.OptionsModel(**manual_problem)
template_code, resolved_problem = hg.generate(options_model, return_selections=True)

manual_template_path = generated_dir / "honegumi_materials_template.py"
manual_template_path.write_text(template_code, encoding="utf-8")

print("Resolved Honegumi selections:")
pprint(resolved_problem)
print(f"\nTemplate written to: {manual_template_path}")
display(Code("\n".join(template_code.splitlines()[:80]), language="python"))


Resolved Honegumi selections:
{'categorical': False,
 'composition_constraint': True,
 'custom_gen': False,
 'custom_threshold': False,
 'dummy': False,
 'existing_data': True,
 'is_compatible': True,
 'linear_constraint': False,
 'model': 'Default',
 'model_kwargs': {},
 'objective': 'Single',
 'order_constraint': False,
 'sum_constraint': False,
 'synchrony': 'Batch',
 'task': 'Single',
 'visualize': True}

Template written to: generated\honegumi_materials_template.py


# Generated by Honegumi (https://arxiv.org/abs/2502.06815)
# %pip install ax-platform==0.4.3 matplotlib
import numpy as np
import pandas as pd
from ax.service.ax_client import AxClient, ObjectiveProperties
import matplotlib.pyplot as plt

obj1_name = "branin"


def branin3(x1, x2, x3):
    y = float(
        (x2 - 5.1 / (4 * np.pi**2) * x1**2 + 5.0 / np.pi * x1 - 6.0) ** 2
        + 10 * (1 - 1.0 / (8 * np.pi)) * np.cos(x1)
        + 10
    )

    # Contrived way to incorporate x3 into the objective
    y = y * (1 + 0.1 * x1 * x2 * x3)

    return y


# Define total for compositional constraint, where x1 + x2 + x3 == total
total = 10.0


# Define the training data

# note that for this training data, the compositional constraint is satisfied


X_train = pd.DataFrame(
    [
        {"x1": 4.0, "x2": 5.0, "x3": 1.0},
        {"x1": 0.0, "x2": 6.2, "x3": 3.8},
        {"x1": 5.9, "x2": 2.0, "x3": 2.0},
        {"x1": 1.5, "x2": 2.0, "x3": 6.5},
        {"x1": 1.0, "x2": 9.0, "x3": 0.0},
    ]
)

# Define y_train (normally the values would be supplied directly instead of calculating here)
y_train = [branin3(row["x1"], row["x2"], row["x3"]) for _, row in X_train.iterrows()]

# See https://youtu.be/4tnaL9ts6CQ for simple human-in-the-loop BO instructions

# Define the number of training examples
n_train = len(X_train)


ax_client = AxClient()

ax_client.create_experiment(
    parameters=[
        {"name": "x1", "type": "range", "bounds": [0.0, total]},
        {"name": "x2", "type": "range", "bounds": [0.0, total]},
    ],
    objectives={
        obj1_name: ObjectiveProperties(minimize=True),
    },
    parameter_constraints=[
        f"x1 + x2 <= {total}",  # reparameterized compositional constraint, which is a type of sum constraint
    ],
)

# Add existing data to the AxClient
for i in range(n_train):
    parameterization = X_train.iloc[i].to_dict()

    # remove x3, since it's hidden from search space due to composition constraint
    parameterization.pop("x3")

    ax_client.attach_trial(parameterization)
    ax_client.complete_trial(trial_index=i, raw_data=y_train[i])


batch_size = 2

In [5]:
scenario_library = pd.DataFrame(
    [
        {
            "Scenario": "Ag nanoparticle recipe tuning",
            "objective": "Single",
            "task": "Single",
            "existing_data": True,
            "composition_constraint": False,
            "categorical": False,
            "synchrony": "Single",
        },
        {
            "Scenario": "Battery cathode composition screening",
            "objective": "Single",
            "task": "Single",
            "existing_data": True,
            "composition_constraint": True,
            "categorical": False,
            "synchrony": "Batch",
        },
        {
            "Scenario": "Perovskite processing with solvent choice",
            "objective": "Single",
            "task": "Single",
            "existing_data": False,
            "composition_constraint": False,
            "categorical": True,
            "synchrony": "Single",
        },
        {
            "Scenario": "Cathode capacity vs. stability tradeoff",
            "objective": "Multi",
            "task": "Single",
            "existing_data": True,
            "composition_constraint": True,
            "categorical": False,
            "synchrony": "Batch",
        },
    ]
)
display(scenario_library)


,Scenario,objective,task,existing_data,composition_constraint,categorical,synchrony
0,Ag nanoparticle recipe tuning,Single,Single,True,False,False,Single
1,Battery cathode composition screening,Single,Single,True,True,False,Batch
2,Perovskite processing with solvent choice,Single,Single,False,False,True,Single
3,Cathode capacity vs. stability tradeoff,Multi,Single,True,True,False,Batch


### Interactive Scenario Picker

This widget is useful for discussion because students can switch between materials scenarios and immediately see how the Honegumi settings and template change.


In [6]:
scenario_map = {
    "Ag nanoparticle recipe tuning": {
        "objective": "Single",
        "model": "Default",
        "task": "Single",
        "categorical": False,
        "sum_constraint": False,
        "order_constraint": False,
        "linear_constraint": False,
        "composition_constraint": False,
        "custom_threshold": False,
        "existing_data": True,
        "synchrony": "Single",
        "visualize": True,
    },
    "Battery cathode composition screening": {
        "objective": "Single",
        "model": "Default",
        "task": "Single",
        "categorical": False,
        "sum_constraint": False,
        "order_constraint": False,
        "linear_constraint": False,
        "composition_constraint": True,
        "custom_threshold": False,
        "existing_data": True,
        "synchrony": "Batch",
        "visualize": True,
    },
    "Perovskite processing with solvent choice": {
        "objective": "Single",
        "model": "Default",
        "task": "Single",
        "categorical": True,
        "sum_constraint": False,
        "order_constraint": False,
        "linear_constraint": False,
        "composition_constraint": False,
        "custom_threshold": False,
        "existing_data": False,
        "synchrony": "Single",
        "visualize": True,
    },
    "Cathode capacity vs. stability tradeoff": {
        "objective": "Multi",
        "model": "Default",
        "task": "Single",
        "categorical": False,
        "sum_constraint": False,
        "order_constraint": False,
        "linear_constraint": False,
        "composition_constraint": True,
        "custom_threshold": True,
        "existing_data": True,
        "synchrony": "Batch",
        "visualize": True,
    },
}


def show_honegumi_scenario(scenario_name):
    scenario = scenario_map[scenario_name]
    scenario_model = hg.OptionsModel(**scenario)
    scenario_code, scenario_resolved = hg.generate(scenario_model, return_selections=True)
    display(pd.DataFrame([scenario_resolved]))
    display(Markdown("**Template preview**"))
    display(Code("\n".join(scenario_code.splitlines()[:60]), language="python"))


widgets.interact(
    show_honegumi_scenario,
    scenario_name=widgets.Dropdown(
        options=list(scenario_map.keys()),
        value="Battery cathode composition screening",
        description="Scenario",
        layout=widgets.Layout(width="70%"),
    ),
)


interactive(children=(Dropdown(description='Scenario', index=1, layout=Layout(width='70%'), options=('Ag nanop…

<function __main__.show_honegumi_scenario(scenario_name)>

## 3. From Natural Language to Honegumi Parameters

The Honegumi RAG Assistant adds an LLM-based parameter selector on top of Honegumi.
The first step is to interpret a free-text problem description and infer the Honegumi settings.

This live step requires an `OPENAI_API_KEY`.


In [7]:
problem_description = (
    "Optimize a nickel-rich cathode synthesis for maximum discharge capacity while keeping "
    "cycling degradation low. We already have historical data from prior formulations, the "
    "composition fractions should add up to the total precursor blend, and we can test four "
    "candidate recipes per experimental round."
)

print(problem_description)


Optimize a nickel-rich cathode synthesis for maximum discharge capacity while keeping cycling degradation low. We already have historical data from prior formulations, the composition fractions should add up to the total precursor blend, and we can test four candidate recipes per experimental round.


In [8]:
if settings.openai_api_key:
    extracted = ParameterExtractor.invoke(problem_description)
    pprint(extracted)
else:
    extracted = {"bo_params": None}
    print("OPENAI_API_KEY is not set, so live parameter extraction is skipped.")
    print("Add the key to a local .env file and rerun this cell to exercise the RAG assistant.")


OPENAI_API_KEY is not set, so live parameter extraction is skipped.
Add the key to a local .env file and rerun this cell to exercise the RAG assistant.


In [9]:
if extracted.get("bo_params"):
    rag_options = hg.OptionsModel(**extracted["bo_params"])
    rag_template_code = hg.generate(rag_options)
    rag_template_path = generated_dir / "honegumi_rag_selected_template.py"
    rag_template_path.write_text(rag_template_code, encoding="utf-8")
    print(f"RAG-selected template written to: {rag_template_path}")
    display(Code("\n".join(rag_template_code.splitlines()[:80]), language="python"))
else:
    print("No extracted parameters available yet.")


No extracted parameters available yet.


## 4. Full Honegumi RAG Workflow

The full pipeline can:

1. infer Honegumi parameters from the problem description,
2. generate a deterministic skeleton,
3. retrieve Ax documentation context from a vector store,
4. write a fuller Bayesian optimization script.

The vector store is optional, but recommended for better retrieval quality.


In [10]:
vectorstore_path = Path(settings.retrieval_vectorstore_path) if settings.retrieval_vectorstore_path else Path("data/processed/ax_docs_vectorstore")
print(f"Configured vector store path: {vectorstore_path}")
print(f"Exists: {vectorstore_path.exists()}")
print("\nRecommended one-time build command:")
print("uv run python -m honegumi_rag_assistant.build_vector_store")


Configured vector store path: data\processed\ax_docs_vectorstore
Exists: False

Recommended one-time build command:
uv run python -m honegumi_rag_assistant.build_vector_store


In [11]:
if settings.openai_api_key:
    try:
        rag_code = run_from_text(
            problem_description,
            output_dir=str(generated_dir),
            debug=False,
            enable_review=False,
        )
        print("Full RAG pipeline completed.")
        display(Code("\n".join(rag_code.splitlines()[:100]), language="python"))
    except Exception as exc:
        print(f"Full RAG pipeline did not complete: {exc}")
        print("If retrieval is the blocker, build the vector store with:")
        print("uv run python -m honegumi_rag_assistant.build_vector_store")
else:
    print("Skipping the live RAG pipeline because OPENAI_API_KEY is not set.")


Skipping the live RAG pipeline because OPENAI_API_KEY is not set.


## 5. Useful Commands Outside the Notebook

```powershell
uv sync
uv run jupyter lab
uv run honegumi-rag --help
uv run python -m honegumi_rag_assistant.build_vector_store
```

Once an API key is present in `.env`, you can also run the assistant directly from the terminal and describe your materials optimization problem interactively.
